In [10]:
import pandas as pd
import re
import emoji
import html
from tqdm import tqdm
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
tqdm.pandas()

print("Library berhasil dimuat")

Library berhasil dimuat


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
df = pd.read_csv("rawData/dataX.csv")
df = df[['created_at', 'user_id_str', 'full_text']]
df['full_text'] = df['full_text'].fillna('')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3714 entries, 0 to 3713
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   created_at   3714 non-null   object
 1   user_id_str  3714 non-null   int64 
 2   full_text    3714 non-null   object
dtypes: int64(1), object(2)
memory usage: 87.2+ KB


In [12]:
initial_count = len(df)
df.drop_duplicates(subset=['full_text'], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Data dimuat: {len(df)} tweet (dihapus {initial_count - len(df)} duplikat)")
print(f"Kolom: {df.columns.tolist()}")

Data dimuat: 1882 tweet (dihapus 1832 duplikat)
Kolom: ['created_at', 'user_id_str', 'full_text']


In [13]:
def clean_tweet_bertopic(text):
    if not isinstance(text, str):
        return ""

    # Decode HTML entities (mis. &amp; → &)
    text = html.unescape(text)

    # Normalkan emoji → teks deskriptif, lalu hapus tag-nya
    text = emoji.demojize(text, language='en')
    text = re.sub(r':[\w_]+:', ' ', text)

    # Hapus URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)

    # Hapus mention (@username)
    text = re.sub(r'@\w+', '', text)

    # Hapus simbol hashtag, pertahankan kata di belakangnya
    text = re.sub(r'#(\w+)', r' \1', text)

    # Hapus indikator retweet
    text = re.sub(r'^RT[\s]+', '', text)

    # Hapus artefak HTML tersisa (mis. &gt; &lt;)
    text = re.sub(r'&\w+;', '', text)

    # Hapus tanda baca, simbol, dan angka
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\d+', ' ', text)

    # Bersihkan spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()

    return text

def case_folding(text):
    if not isinstance(text, str):
        return ""
    return text.lower()

# Kolom data_cleaning: hasil bersih sebelum case folding
df['data_cleaning'] = df['full_text'].progress_apply(clean_tweet_bertopic)

# Kolom case_folding: hasil setelah diubah ke huruf kecil
df['case_folding'] = df['data_cleaning'].progress_apply(case_folding)

print("Cleaning dan case folding selesai")

100%|██████████| 1882/1882 [00:00<00:00, 980581.38it/s]

Cleaning dan case folding selesai


In [14]:
kamus_alay = pd.read_csv(
    "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
)

slang_dict = (
    kamus_alay.drop_duplicates(subset=['slang'])
    .set_index('slang')['formal']
    .to_dict()
)

manual_dict = {
    'yg'    : 'yang',
    'dg'    : 'dengan',
    'dgn'   : 'dengan',
    'tdk'   : 'tidak',
    'krn'   : 'karena',
    'sdh'   : 'sudah',
    'udh'   : 'sudah',
    'bs'    : 'bisa',
    'utk'   : 'untuk',
    'bgt'   : 'sangat',
    'banget': 'sangat',
    'tp'    : 'tapi',
    'skrng' : 'sekarang',
    'skrg'  : 'sekarang',
    'tau'   : 'tahu',
    'gk'    : 'tidak',
    'gak'   : 'tidak',
    'ga'    : 'tidak',
    'kpd'   : 'kepada',
    'tsb'   : 'tersebut',
    'ttg'   : 'tentang',
    'thd'   : 'terhadap',
    'dr'    : 'dari',
    'pd'    : 'pada',
    'dlm'   : 'dalam',
    'utk'   : 'untuk',
    'jd'    : 'jadi',
    # Partikel penegas → hapus
    'deh'   : '',
    'sih'   : '',
    'dong'  : '',
    'kok'   : '',
    'loh'   : '',
    'lho'   : '',
    'nah'   : '',
    'nih'   : '',
    'tuh'   : '',
}

# Merge: manual_dict override kamus alay jika ada konflik
slang_dict.update(manual_dict)

print(f"Kamus alay dimuat : {len(kamus_alay)} entri")
print(f"Setelah merge     : {len(slang_dict)} entri unik")

Kamus alay dimuat : 15006 entri
Setelah merge     : 4343 entri unik


In [15]:
def normalize_with_kamus(text):
    if not isinstance(text, str):
        return ""

    words = text.split()
    normalized = []
    for word in words:
        replacement = slang_dict.get(word)
        if replacement is None:
            normalized.append(word)       # tidak ada di kamus → pertahankan
        elif replacement:
            normalized.append(replacement) # ada pengganti → ganti
        # else: pengganti kosong (partikel) → buang

    return ' '.join(normalized)

df['normalization'] = df['case_folding'].progress_apply(normalize_with_kamus)
print("Normalisasi dengan kamus alay selesai")

100%|██████████| 1882/1882 [00:00<00:00, 188501.29it/s]

Normalisasi dengan kamus alay selesai


In [16]:
from Sastrawi.StopWordRemover.StopWordRemoverFactory import (
    StopWordRemoverFactory, StopWordRemover, ArrayDictionary
)

sastrawi_stopwords = StopWordRemoverFactory().get_stop_words()

custom_stopwords = [
    # Noise yang masih muncul di topic word scores
    'yang', 'gk', 'tau', 'skrng', 'skrg',

    # Partikel informal Twitter yang lolos normalisasi
    'deh', 'sih', 'dong', 'kok', 'loh', 'lho',
    'nah', 'nih', 'tuh', 'gitu', 'gituh', 'gini',
    'wah', 'eh', 'ah', 'oh', 'ih', 'yah', 'aduh',

    # Kata umum rendah makna di konteks Twitter
    'bilang', 'kayak', 'kayaknya', 'emang',
    'kaya', 'udah', 'uda', 'udh',

    # Kata generik yang tidak berkontribusi ke topik BBM/etanol
    'orang', 'banyak', 'lebih', 'sama', 'bisa',
    'hal', 'cara', 'jadi', 'buat', 'bikin',
]

# Gabungkan stopword Sastrawi + custom
combined_stopwords = sastrawi_stopwords + custom_stopwords
dictionary = ArrayDictionary(combined_stopwords)
stopword_remover = StopWordRemover(dictionary)

def remove_stopwords_x(text):
    if not isinstance(text, str):
        return ""
    return stopword_remover.remove(text)

df['stopword_removal'] = df['normalization'].progress_apply(remove_stopwords_x)

# Filter tweet terlalu pendek setelah stopword removal
min_length = 15
initial_len = len(df)
df = df[df['stopword_removal'].str.len() >= min_length]
df.reset_index(drop=True, inplace=True)

print("Stopword removal selesai")
print(f"Stopword Sastrawi  : {len(sastrawi_stopwords)} kata")
print(f"Custom stopword    : {len(custom_stopwords)} kata")
print(f"Total gabungan     : {len(combined_stopwords)} kata")
print(f"Tweet dihapus      : {initial_len - len(df)} (panjang < {min_length} karakter)")
print(f"Tweet tersisa      : {len(df)}")

100%|██████████| 1882/1882 [00:00<00:00, 169454.10it/s]

Stopword removal selesai
Stopword Sastrawi  : 809 kata
Custom stopword    : 42 kata
Total gabungan     : 851 kata
Tweet dihapus      : 34 (panjang < 15 karakter)
Tweet tersisa      : 1848


In [17]:
print("=" * 60)
print("LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)")
print("=" * 60)

df['orig_length']  = df['full_text'].str.len()
df['final_length'] = df['stopword_removal'].str.len()
df['orig_words']   = df['full_text'].apply(lambda x: len(str(x).split()))
df['final_words']  = df['stopword_removal'].apply(lambda x: len(str(x).split()))

avg_orig  = df['orig_length'].mean()
avg_final = df['final_length'].mean()
avg_orig_w  = df['orig_words'].mean()
avg_final_w = df['final_words'].mean()

print(f"\nStatistik panjang teks:")
print(f"  Rata-rata panjang asli  : {avg_orig:.1f} karakter")
print(f"  Rata-rata panjang akhir : {avg_final:.1f} karakter")
print(f"  Reduksi                 : {((avg_orig - avg_final) / avg_orig * 100):.1f}%")

print(f"\nStatistik jumlah kata:")
print(f"  Rata-rata kata asli  : {avg_orig_w:.1f}")
print(f"  Rata-rata kata akhir : {avg_final_w:.1f}")
print(f"  Reduksi              : {((avg_orig_w - avg_final_w) / avg_orig_w * 100):.1f}%")

print(f"\nDistribusi panjang teks akhir:")
print(f"  Terpendek : {df['final_length'].min()} karakter")
print(f"  Terpanjang: {df['final_length'].max()} karakter")
print(f"  Median    : {df['final_length'].median():.1f} karakter")

LAPORAN KUALITAS PREPROCESSING — DATA X (BERTopic)

Statistik panjang teks:
  Rata-rata panjang asli  : 170.0 karakter
  Rata-rata panjang akhir : 100.2 karakter
  Reduksi                 : 41.1%

Statistik jumlah kata:
  Rata-rata kata asli  : 24.3
  Rata-rata kata akhir : 13.9
  Reduksi              : 42.9%

Distribusi panjang teks akhir:
  Terpendek : 15 karakter
  Terpanjang: 260 karakter
  Median    : 95.0 karakter


In [18]:
bertopic_data = df[[
    'created_at',
    'user_id_str',
    'full_text',         # teks asli
    'data_cleaning',     # setelah cleaning
    'case_folding',      # setelah huruf kecil
    'normalization',     # setelah normalisasi kata
    'stopword_removal',
]].copy()

bertopic_data['preprocessing_stats'] = bertopic_data.apply(
    lambda row: f"orig:{len(row['full_text'])} -> final:{len(row['normalization'])}",
    axis=1
)

output_file = "cleanData/bertopic_ready_x.csv"
bertopic_data.to_csv(output_file, index=False, encoding='utf-8')

print(f"Data diekspor ke  : {output_file}")
print(f"Jumlah tweet siap : {len(bertopic_data)}")
print(f"Kolom             : {bertopic_data.columns.tolist()}")

Data diekspor ke  : cleanData/bertopic_ready_x.csv
Jumlah tweet siap : 1848
Kolom             : ['created_at', 'user_id_str', 'full_text', 'data_cleaning', 'case_folding', 'normalization', 'stopword_removal', 'preprocessing_stats']
